# Held-out evaluation
Research only, not clinical diagnosis. Pima cohort is not representative of the general population. Raw dataset is not synthetic; never use it as demo patient data. See docs/data_card.md. Fixed seed=42; train/validation/test=70/15/15. Test never used for fitting or model selection.

In [1]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT/'pyproject.toml').exists(): ROOT=ROOT.parent
assert (ROOT/'pyproject.toml').exists()
sys.path.insert(0,str(ROOT))
import matplotlib
matplotlib.use('Agg')
from backend.app.ml.pipeline import *
import matplotlib.pyplot as plt


In [2]:
def evaluate():
    import matplotlib.pyplot as plt
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,average_precision_score,brier_score_loss,ConfusionMatrixDisplay,RocCurveDisplay,PrecisionRecallDisplay
    from sklearn.calibration import CalibrationDisplay
    seed_all(); parts=arrays(); x,y=parts['train']; tx,ty=parts['test']
    model=load_checkpoint()
    with torch.no_grad(): scores={'MLP':torch.sigmoid(model(torch.from_numpy(tx))).numpy()}
    for name,baseline in [('LogisticRegression',LogisticRegression(max_iter=2000,random_state=SEED)),('RandomForest',RandomForestClassifier(n_estimators=300,min_samples_leaf=2,random_state=SEED,n_jobs=1))]:
        baseline.fit(x,y); scores[name]=baseline.predict_proba(tx)[:,1]
    metrics={}
    for name,score in scores.items():
        pred=(score>=0.5).astype(int)
        metrics[name]={'accuracy':accuracy_score(ty,pred),'precision':precision_score(ty,pred,zero_division=0),'recall':recall_score(ty,pred,zero_division=0),'f1':f1_score(ty,pred,zero_division=0),'roc_auc':roc_auc_score(ty,score),'pr_auc_average_precision':average_precision_score(ty,score),'brier_score':brier_score_loss(ty,score)}
        fig,axes=plt.subplots(2,2,figsize=(10,8))
        ConfusionMatrixDisplay.from_predictions(ty,pred,ax=axes[0,0])
        RocCurveDisplay.from_predictions(ty,score,ax=axes[0,1])
        PrecisionRecallDisplay.from_predictions(ty,score,ax=axes[1,0])
        CalibrationDisplay.from_predictions(ty,score,n_bins=5,strategy='quantile',ax=axes[1,1]); axes[1,1].plot([0,1],[0,1],'k--')
        fig.suptitle(name+' â€” held-out test, research only'); fig.tight_layout(); fig.savefig(FIG/(name+'_evaluation.png')); plt.show(); plt.close(fig)
        save_json(RES/(name+'_errors.json'),{'false_positive_indices':np.array(json.loads((RES/'splits.json').read_text())['test'])[(pred==1)&(ty==0)].tolist(),'false_negative_indices':np.array(json.loads((RES/'splits.json').read_text())['test'])[(pred==0)&(ty==1)].tolist()})
    save_json(RES/'metrics.json',metrics); frame=pd.DataFrame(metrics).T; frame.to_csv(RES/'metrics.csv',index_label='model')
    print('No test-driven threshold tuning. Calibration is assessed, not fitted on test.')
    return frame


In [3]:
metrics=evaluate()
display(metrics)

D:\TEMP\ipykernel_7484\641884857.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.suptitle(name+' â€” held-out test, research only'); fig.tight_layout(); fig.savefig(FIG/(name+'_evaluation.png')); plt.show(); plt.close(fig)


D:\TEMP\ipykernel_7484\641884857.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.suptitle(name+' â€” held-out test, research only'); fig.tight_layout(); fig.savefig(FIG/(name+'_evaluation.png')); plt.show(); plt.close(fig)


No test-driven threshold tuning. Calibration is assessed, not fitted on test.


D:\TEMP\ipykernel_7484\641884857.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.suptitle(name+' â€” held-out test, research only'); fig.tight_layout(); fig.savefig(FIG/(name+'_evaluation.png')); plt.show(); plt.close(fig)


,accuracy,precision,recall,f1,roc_auc,pr_auc_average_precision,brier_score
MLP,0.758621,0.709677,0.536585,0.611111,0.863415,0.781957,0.151729
LogisticRegression,0.775862,0.777778,0.512195,0.617647,0.862439,0.783225,0.149270
RandomForest,0.732759,0.666667,0.487805,0.563380,0.827967,0.757576,0.161513


Limitations: small cohort, female Pima ancestry age >=21; historical diagnostic labels, missingness and potential selection bias. Single split has high uncertainty. Brier/reliability curve assess uncalibrated scores; no clinical threshold or external validation. Baselines use identical train-only preprocessing and held-out test; DL need not outperform RF. False positives/negatives are exported for error review without copying patient rows.